<a href="https://colab.research.google.com/github/deetijasmitha/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/deetijasmitha/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

### Finding 1 — CTR changes with search position

The paper reports that CTR is higher for pages in better search positions and drops as position gets worse. The label for this finding comes from observed CTR grouped by search-position tiers.

My methodology question is whether the comparison is descriptive rather than causal. The validation design supports the observed relationship, but it does not prove that improving position will directly cause CTR to increase.

### Finding 2 — Content age is related to performance

The paper reports differences in performance across content-age or freshness groups. The label comes from observed search-performance measurements for pages in different age groups.

My methodology question is whether content age itself causes the performance difference. The validation can show an observed association between age and performance, but other factors may also contribute. Therefore, the finding should be treated as directional evidence rather than a causal claim.

In [6]:
# Section 1 — simple methodology check

print("Finding 1: CTR by search-position tier")
print("Label/measure: observed CTR")
print("Validation question: Does the design show association or causation?")
print()

print("Finding 2: Performance by content-age/freshness group")
print("Label/measure: observed search-performance metrics")
print("Validation question: Could other factors explain the observed difference?")

Finding 1: CTR by search-position tier
Label/measure: observed CTR
Validation question: Does the design show association or causation?

Finding 2: Performance by content-age/freshness group
Label/measure: observed search-performance metrics
Validation question: Could other factors explain the observed difference?


## 2. My model under an honest split (before/after)

My Week-5 Logistic Regression used a client-grouped split, with 20% of clients held out for testing.

The Week-5 model achieved a ROC-AUC of 0.7056.

I re-run the same model and split design here to verify the result. The grouped split keeps clients separated between training and testing, making the validation more honest for this dataset.

Before: ROC-AUC = 0.7056

After: ROC-AUC will be calculated again below.

In [7]:
# Section 2 — Re-run Week-5 model under client-grouped validation

import duckdb
from sklearn.model_selection import GroupShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# ---------------------------------------------------------
# 1. Connect to the existing warehouse
# ---------------------------------------------------------

con = duckdb.connect()

# Use the existing HF token from Week-5
try:
    HF_TOKEN
except NameError:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")

try:
    con.execute("DROP SECRET hf_token")
except:
    pass

con.execute("""
CREATE SECRET hf_token (
    TYPE HUGGINGFACE,
    TOKEN ?
)
""", [HF_TOKEN])

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"

FEB = f"{FACT}/month=2026-02/*.parquet"
MAR = f"{FACT}/month=2026-03/*.parquet"

# ---------------------------------------------------------
# 2. Build February features
# ---------------------------------------------------------

feb_query = f"""
SELECT
    client_hash_id,
    content_hash_id,

    SUM(gsc_impressions) AS gsc_impressions_feb,
    SUM(gsc_clicks) AS gsc_clicks_feb,
    AVG(gsc_avg_position) AS gsc_avg_position_feb,

    SUM(ga4_pageviews) AS ga4_pageviews_feb,
    SUM(ga4_sessions) AS ga4_sessions_feb,
    SUM(ga4_users) AS ga4_users_feb,
    SUM(ga4_engaged_sessions) AS ga4_engaged_sessions_feb,
    SUM(ga4_total_engagement_sec) AS ga4_total_engagement_sec_feb

FROM read_parquet('{FEB}')
WHERE gsc_data_available IS TRUE

GROUP BY
    client_hash_id,
    content_hash_id

HAVING
    SUM(gsc_impressions) >= 100
    AND SUM(gsc_clicks) >= 3
"""

feb_df = con.sql(feb_query).df()

# ---------------------------------------------------------
# 3. Build March outcome
# ---------------------------------------------------------

mar_query = f"""
SELECT
    client_hash_id,
    content_hash_id,

    SUM(gsc_clicks) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS gsc_clicks_mar,

    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS measured_days_mar

FROM read_parquet('{MAR}')

GROUP BY
    client_hash_id,
    content_hash_id
"""

mar_df = con.sql(mar_query).df()

# ---------------------------------------------------------
# 4. Merge February features with March outcome
# ---------------------------------------------------------

model_df = feb_df.merge(
    mar_df,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

model_df["gsc_clicks_mar"] = model_df["gsc_clicks_mar"].fillna(0)
model_df["measured_days_mar"] = model_df["measured_days_mar"].fillna(0)

# Target: March clicks equal to zero
model_df["went_dark"] = (
    model_df["gsc_clicks_mar"] == 0
).astype(int)

# ---------------------------------------------------------
# 5. Client-grouped split
# ---------------------------------------------------------

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        model_df,
        model_df["went_dark"],
        groups=model_df["client_hash_id"]
    )
)

train_df = model_df.iloc[train_idx].copy()
test_df = model_df.iloc[test_idx].copy()

# ---------------------------------------------------------
# 6. Features — exactly the same as Week-5
# ---------------------------------------------------------

FEATURE_COLUMNS = [
    "gsc_impressions_feb",
    "gsc_clicks_feb",
    "gsc_avg_position_feb",
    "ga4_pageviews_feb",
    "ga4_sessions_feb",
    "ga4_users_feb",
    "ga4_engaged_sessions_feb",
    "ga4_total_engagement_sec_feb"
]

X_train = train_df[FEATURE_COLUMNS].copy()
y_train = train_df["went_dark"].copy()

X_test = test_df[FEATURE_COLUMNS].copy()
y_test = test_df["went_dark"].copy()

# ---------------------------------------------------------
# 7. Imputation and scaling
# ---------------------------------------------------------

imputer = SimpleImputer(strategy="median")

X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

# ---------------------------------------------------------
# 8. Train Logistic Regression
# ---------------------------------------------------------

logistic_model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42
)

logistic_model.fit(
    X_train_scaled,
    y_train
)

# ---------------------------------------------------------
# 9. Evaluate
# ---------------------------------------------------------

model_probabilities = logistic_model.predict_proba(
    X_test_scaled
)[:, 1]

honest_auc = roc_auc_score(
    y_test,
    model_probabilities
)

# ---------------------------------------------------------
# 10. Validation checks
# ---------------------------------------------------------

train_clients = set(train_df["client_hash_id"])
test_clients = set(test_df["client_hash_id"])

client_overlap = train_clients & test_clients

print("========== SECTION 2 CHECK ==========")
print("Feature window: February 2026")
print("Outcome window: March 2026")
print("Split type: client-grouped")
print()
print("Training rows:", len(train_df))
print("Test rows:", len(test_df))
print()
print("Training clients:", train_df["client_hash_id"].nunique())
print("Test clients:", test_df["client_hash_id"].nunique())
print("Client overlap:", len(client_overlap))
print()
print("Before ROC-AUC: 0.7056")
print("After ROC-AUC:", round(honest_auc, 4))
print()

assert len(client_overlap) == 0
assert 0 <= honest_auc <= 1

print("SECTION 2 CHECK: PASS")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

========== SECTION 2 CHECK ==========
Feature window: February 2026
Outcome window: March 2026
Split type: client-grouped

Training rows: 22080
Test rows: 7649

Training clients: 25
Test clients: 7
Client overlap: 0

Before ROC-AUC: 0.7056
After ROC-AUC: 0.7056

SECTION 2 CHECK: PASS


## 3. Leakage audit

I checked the final feature set used by the model for target leakage.

The model uses only February 2026 information to predict the March 2026 `went_dark` outcome. The March outcome is not included in the input features.

The feature set therefore respects the time boundary and does not use the target or future information as a feature.

In [8]:
# Section 3 — Leakage audit

TARGET = "went_dark"

future_or_target_columns = [
    col for col in FEATURE_COLUMNS
    if col == TARGET
    or "mar" in col.lower()
    or "march" in col.lower()
]

print("========== SECTION 3 LEAKAGE AUDIT ==========")
print("Target:", TARGET)
print("Final features:")
for col in FEATURE_COLUMNS:
    print("-", col)

print()
print("Potential target/future leakage columns:", future_or_target_columns)

if len(future_or_target_columns) == 0:
    print()
    print("LEAKAGE AUDIT: PASS")
    print("All model features come from the February feature window.")
else:
    print()
    print("LEAKAGE AUDIT: REVIEW REQUIRED")

========== SECTION 3 LEAKAGE AUDIT ==========
Target: went_dark
Final features:
- gsc_impressions_feb
- gsc_clicks_feb
- gsc_avg_position_feb
- ga4_pageviews_feb
- ga4_sessions_feb
- ga4_users_feb
- ga4_engaged_sessions_feb
- ga4_total_engagement_sec_feb

Potential target/future leakage columns: []

LEAKAGE AUDIT: PASS
All model features come from the February feature window.


## 4. Claim rewrite

My original result could be overstated if described as proof that the model will predict future content performance.

A safer claim is:

The Logistic Regression model measured a ROC-AUC of 0.7056 on a client-grouped test set. This provides directional evidence that February performance features can help identify content that went dark in March. The result is decision-support evidence, not a causal claim or a guarantee of future performance.

In [9]:
# Section 4 — Claim audit check

print("========== SECTION 4 CLAIM CHECK ==========")

claim_words = [
    "measured",
    "directional",
    "decision-support"
]

safe_claim = (
    "The Logistic Regression model measured a ROC-AUC of 0.7056 "
    "on a client-grouped test set. This provides directional evidence "
    "that February performance features can help identify content that "
    "went dark in March. The result is decision-support evidence, "
    "not a causal claim or a guarantee of future performance."
)

print(safe_claim)
print()

for word in claim_words:
    print(f"{word}: {'PASS' if word in safe_claim else 'CHECK'}")

print()
print("SECTION 4 CHECK: PASS")

========== SECTION 4 CLAIM CHECK ==========
The Logistic Regression model measured a ROC-AUC of 0.7056 on a client-grouped test set. This provides directional evidence that February performance features can help identify content that went dark in March. The result is decision-support evidence, not a causal claim or a guarantee of future performance.

measured: PASS
directional: PASS
decision-support: PASS

SECTION 4 CHECK: PASS


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors
- [x] No client names, URLs, or private queries are included
- [x] Claims use careful words: observed, measured, directional, decision-support
- [x] The model uses a client-grouped validation split
- [x] Training and test clients have 0 overlap
- [x] Leakage audit passed with 0 potential target/future leakage columns
- [x] Week-5 ROC-AUC: 0.7056
- [x] Honest grouped-split ROC-AUC: 0.7056
- [x] Committed to the repository under `work/notebooks/`
- [x] Ready to submit the repository URL on the FlyRank card